In [1]:
from pymongo import MongoClient

cliente= MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)
print(cliente.admin.command("ping"))

db= cliente["panteon"]

{'ok': 1.0}


## Configuración inicial y librerías
Se importa pandas y se definen las rutas de los archivos

In [2]:
import pandas as pd

columnas =['Kingdom','Phylum','Class','Order','Family','Species','Source','Source 2','Notes']
nulos =['NA','N/A','','-']


ruta_default_csv ="PanTEon_Database_metadata_v1.6.1.csv"
ruta_default_fasta ="PanTEon_Database_v1.6.1.fasta"


## Leer y limpiar el csv
Funciones para leer el CSV con la taxonomía, para cambiarle los nombres a las columnas en español y quitar filas vacías que dañen el análisis.   

In [4]:

def leer_csv(ruta):
    try:
        especies_df =pd.read_csv(ruta,usecols=columnas,na_values=nulos)

        columnas_texto = especies_df.select_dtypes(
            include=["object", "string"]
        ).columns

        for columna in columnas_texto:
            especies_df[columna] =(especies_df[columna].str.strip().str.replace(r"\s+", " ", regex=True))

        return especies_df
    except FileNotFoundError:
        print(f"No se encontró el archivo '{ruta}'.")
    except pd.errors.ParserError:
        print(f"N se pudo interpretar el csv '{ruta}'.")
    except Exception as e:
        print(f"error inesperado al leer el csv: {e}")

def renombrar_columnas(especies_df):
    especies_df_renombrado =especies_df.rename(
        columns={
            'Kingdom':'reino',
            'Phylum':'filo',
            'Class':'tax_clase',
            'Order':'tax_orden',
            'Family':'familia',
            'Species':'especie',
            'Source':'fuente',
            'Source 2':'fuente_2',
            'Notes': 'notas'
        }
    )

    return especies_df_renombrado

def tratar_faltantes(especies_df):
    especies_df_limpio =especies_df.copy()
    especies_df_limpio =especies_df_limpio.dropna(subset=['especie'])

    especies_df_limpio  =especies_df_limpio.astype(object).where(especies_df_limpio.notna(),None)

    return especies_df_limpio



# Funciones del fasta y cruce
Funciones para leer el fasta y comparar los datos.

In [5]:
def leer_especies_fasta(ruta):
    especies ={}
    total_encabezados =0
    sin_especie =[]

    with open(ruta, "r",encoding="utf-8") as archivo:
        for linea in archivo:
            if not linea.startswith(">"):
                continue

            total_encabezados += 1
            cabecera = linea[1:].strip()

            if "@" not in cabecera:
                sin_especie.append(cabecera)
                continue

            nombre =cabecera.rsplit("@",1)[-1].strip()
            clave =normalizar_especie(nombre)

            if clave is None:
                sin_especie.append(cabecera)
                continue

            if clave not in especies:
                especies[clave] ={"nombres":[],"conteo":0}

            if nombre not in especies[clave]["nombres"]:
                especies[clave]["nombres"].append(nombre)

            especies[clave]["conteo"] +=1

    return especies,total_encabezados,sin_especie


def cruzar(df, especies_fasta):
    csv ={}

    for nombre in df["especie"]:
        clave =normalizar_especie(nombre)
        if clave is None:
            continue
        if clave not in csv:
            csv[clave] =[]
        csv[clave].append(nombre)

    claves_csv =set(csv)
    claves_fasta =set(especies_fasta)

    ambos =claves_csv & claves_fasta
    solo_fasta =claves_fasta-claves_csv
    solo_csv =claves_csv-claves_fasta

    return ambos,solo_fasta,solo_csv, csv


def reportar_cruce(ambos, solo_fasta, solo_csv, csv, fasta, total_encabezados, sin_especie, filas_csv):
    print("Filas del csv:",filas_csv)
    print("Especies distintas en el csv:",len(csv))
    print("Encabezados en el fasta:",total_encabezados)
    print("Encabezados sin especie:",len(sin_especie))
    print("Especies distintas en el fasta:",len(fasta))
    print("En los dos:", len(ambos))
    print("Solo en el fasta:",len(solo_fasta))
    print("Solo en el csv:",len(solo_csv))

## Correr las funciones para el csv
Se carga el csv, se renombran columnas y se limpian los nulos.   

In [7]:
especies =leer_csv(ruta_default_csv)
especies =renombrar_columnas(especies)
especies =tratar_faltantes(especies)

## Normalización de nombres

Esta función quita espacios al principio, al final y espacios repetidos de por medio, pasa todo a minúsculas y arregla algunas abreviaturas para que coincidan el csv y el fasta.

In [8]:
def normalizar_especie(nombre):
    if pd.isna(nombre):
        return None

    nombre = str(nombre).strip().lower()
    nombre = " ".join(nombre.split())

    abreviaturas = ["subsp.", "sp.", "var.", "f."]
    palabras = nombre.split()

    for i in range(len(palabras)):
        if palabras[i] in abreviaturas:
            palabras[i] = palabras[i][:-1]

    nombre = " ".join(palabras)

    return nombre or None


## Comparar el csv con el fasta
Se revisa si los nombres de las especies coinciden exactamente entre ambos archivos y si hay diferencias, se reemplazan las del csv por las del fasta 

In [9]:
def revisar_diferencias(ambos, csv, fasta):
    for clave in sorted(ambos):
        nombres_csv =csv[clave]
        nombres_fasta =fasta[clave]["nombres"]
        if not (set(nombres_csv) & set(nombres_fasta)):
            print("csv:", nombres_csv," - fasta",nombres_fasta," - ",fasta[clave]["conteo"],"secuencias")


def reemplazar_nombres(especies, fasta):
    reemplazos =[]
    nombres_nuevos =[]
    for nombre in especies["especie"]:
        clave = normalizar_especie(nombre)
        nombres_fasta = fasta[clave]["nombres"]
        assert len(nombres_fasta)==1
        if nombre in nombres_fasta:
            nombres_nuevos.append(nombre)
        else:
            reemplazos.append((nombre, nombres_fasta[0]))
            nombres_nuevos.append(nombres_fasta[0])

    especies["especie"] = nombres_nuevos
    print(reemplazos)

    return especies

## Ejecutar el cruce y mostrar resultados
Se corren las funciones para leer y cruzar y el print nos muestra cuántas especies hay en cada lado, los iguales y las diferencias de nombres que tocó ajustar.   

In [10]:
fasta,total_encabezados,sin_especie =leer_especies_fasta(ruta_default_fasta)
ambos,solo_fasta,solo_csv,csv = cruzar(especies,fasta)

reportar_cruce(ambos, solo_fasta,solo_csv,csv,fasta,total_encabezados,sin_especie,len(especies))
revisar_diferencias(ambos,csv,fasta)
especies =reemplazar_nombres(especies,fasta)

Filas del csv: 2790
Especies distintas en el csv: 2790
Encabezados en el fasta: 239946
Encabezados sin especie: 0
Especies distintas en el fasta: 2790
En los dos: 2790
Solo en el fasta: 0
Solo en el csv: 0
csv: ['Abrus pulchellus subsp cantoniensis']  - fasta ['Abrus pulchellus subsp. cantoniensis']  -  178 secuencias
csv: ['Beta vulgaris subsp vulgaris']  - fasta ['Beta vulgaris subsp. vulgaris']  -  296 secuencias
csv: ['Coccomyxa sp obi']  - fasta ['Coccomyxa sp. obi']  -  11 secuencias
csv: ['Marchantia polymorpha subsp montivagans']  - fasta ['Marchantia polymorpha subsp. montivagans']  -  48 secuencias
[('Abrus pulchellus subsp cantoniensis', 'Abrus pulchellus subsp. cantoniensis'), ('Beta vulgaris subsp vulgaris', 'Beta vulgaris subsp. vulgaris'), ('Coccomyxa sp obi', 'Coccomyxa sp. obi'), ('Marchantia polymorpha subsp montivagans', 'Marchantia polymorpha subsp. montivagans')]


## Guardar en la base de datos

Se limpia la colección anterior en MongoDB y se mete todo el dataframe ya procesado como documentos nuevos.

In [11]:
def cargar_especies(especies_df, db):
    coleccion =db["especies"]

    documentos = especies_df.to_dict(orient="records")

    coleccion.delete_many({})

    if documentos:
        coleccion.insert_many(documentos)

cargar_especies(especies, db)